# நலம் (Nalam) — Gemma 4 Backend
### Newborn Danger Sign Assessment for ASHA Workers in Tamil Nadu
---

In [ ]:
# Cell 1 — Force Single GPU BEFORE any imports
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("Single GPU mode: ON")

In [ ]:
# Cell 2 — Install dependencies (pinned for Gemma 3n)
# Force-reinstall Pillow first: Kaggle's pre-installed torchvision/PIL
# combination has a known _Ink import mismatch that breaks transformers.
!pip install -q -U --force-reinstall "pillow>=11.0,<12.0"
!pip install -q -U "transformers>=4.53.0" "accelerate>=0.34" "timm"
!pip install -q torch librosa soundfile
!apt-get install -q ffmpeg
print("Dependencies installed — RESTART KERNEL NOW")


In [ ]:
# Cell 3 — Login to HuggingFace
from huggingface_hub import login
login(token="your_hf_token_here")  # Replace with your actual token
print("HuggingFace login successful")

In [ ]:
# Cell 4 — Load Gemma 3n E2B (multimodal: text + audio + image)
# Class: AutoModelForImageTextToText is the right loader for Gemma 3n.
# We load straight to GPU; bfloat16 fits comfortably on a T4 with E2B (~6 GB).

import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "google/gemma-3n-E2B-it"

print(f"Loading processor for {MODEL_ID}...")
processor = AutoProcessor.from_pretrained(MODEL_ID)

print("Loading model (this takes ~60s on first run, model is ~6 GB)...")
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
    low_cpu_mem_usage=True,
)
model.eval()

print(f"Model device: {next(model.parameters()).device}")
print(f"Gemma 3n loaded successfully: {MODEL_ID}")


In [ ]:
# Cell 5 — Helper: Run inference safely on single GPU
def run_inference(messages, max_new_tokens=300):
    """
    Runs Gemma 4 inference ensuring all tensors stay on cuda.
    Handles both text-only and multimodal (audio) inputs.
    """
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True
    )
    
    # Move ALL tensors to cuda explicitly — this is the key fix
    inputs = {
        k: v.to("cuda") if isinstance(v, torch.Tensor) else v
        for k, v in inputs.items()
    }
    
    input_len = inputs["input_ids"].shape[-1]
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )
    
    response = processor.decode(
        outputs[0][input_len:],
        skip_special_tokens=True
    )
    
    return response

print("Inference helper ready")

In [ ]:
# Cell 6 — Test Basic Tamil Text Output
import re

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": """You are a health assistant for ASHA workers in Tamil Nadu.
Write ONE short counselling message in Tamil for a mother whose Day 3 newborn baby is healthy.
Only Tamil. No English. No options. Maximum 3 sentences."""
            }
        ]
    }
]

response = run_inference(messages, max_new_tokens=200)
print("Basic Tamil test:")
print(response)

In [ ]:
# Cell 7 — Feature A1: Safe Visit Counselling (Visit Day Adaptive)
def generate_tamil_counselling(visit_day, baby_weight_kg, is_premature=False, home_delivery=False):
    """
    Generates visit-day specific Tamil counselling for healthy babies.
    Different advice for each visit day based on developmental stage.
    """
    visit_focus = {
        1:  "initiating breastfeeding, skin to skin warmth, cord care, no bathing, colostrum importance",
        3:  "exclusive breastfeeding every 2 hours, cord care check for smell or pus, jaundice signs, warmth",
        7:  "jaundice check, weight regain, breastfeeding 8+ times daily, cord falling off, infection signs",
        14: "BCG and OPV immunization check, growth monitoring, exclusive breastfeeding, mother mental health",
        21: "continued breastfeeding, social smiling development, immunization check, mother nutrition",
        28: "one month immunization due, weight check, exclusive breastfeeding until 6 months, development",
        42: "final HBNC visit, all immunizations check, family planning for mother, continued breastfeeding"
    }
    
    available_days = list(visit_focus.keys())
    closest_day = min(available_days, key=lambda x: abs(x - visit_day))
    focus = visit_focus[closest_day]
    
    risk_notes = []
    if baby_weight_kg < 2.0:
        risk_notes.append(f"LOW BIRTH WEIGHT baby ({baby_weight_kg}kg): emphasize kangaroo mother care and frequent small feeds")
    if is_premature:
        risk_notes.append("PREMATURE baby: emphasize extra warmth and skin to skin contact")
    if home_delivery:
        risk_notes.append("HOME DELIVERY: watch extra carefully for cord infection")
    
    risk_context = " | ".join(risk_notes) if risk_notes else "no additional risk factors"
    
    prompt = f"""You are a health assistant for ASHA workers in rural Tamil Nadu, India.

Day {visit_day} postnatal visit. Baby weight: {baby_weight_kg}kg. {risk_context}
Key topics for this visit: {focus}

Write EXACTLY ONE counselling message in Tamil for the mother.
- Specific to Day {visit_day} — not generic
- Address the most important 2-3 topics for this visit day
- Use simple informal Tamil a village mother understands
- Warm and encouraging tone
- Maximum 5 sentences
- If risk factors exist, address them specifically

Only Tamil. No English. No options. No translations."""

    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    return run_inference(messages, max_new_tokens=300)

# Test
print("TEST — Day 3, Low Birth Weight")
print(generate_tamil_counselling(3, 1.9, home_delivery=True))
print()
print("TEST — Day 14, Normal Baby")
print(generate_tamil_counselling(14, 3.1))

In [ ]:
# Cell 8 — Feature A2: Danger Referral Explanation
def generate_referral_explanation(danger_signs_found):
    """
    Generates calm Tamil explanation when baby needs urgent referral.
    Specific to which danger signs were triggered.
    """
    sign_descriptions = {
        "feeding":   "குழந்தை சரியாக பால் குடிக்கவில்லை",
        "activity":  "குழந்தை சுறுசுறுப்பாக இல்லை",
        "warmth":    "குழந்தையின் உடல் குளிர்ச்சியாக உள்ளது",
        "breathing": "குழந்தை சரியாக மூச்சு விடவில்லை"
    }
    
    signs_tamil = " மற்றும் ".join([
        sign_descriptions[s] for s in danger_signs_found if s in sign_descriptions
    ])
    
    prompt = f"""You are a health assistant for ASHA workers in rural Tamil Nadu.

Danger signs found in newborn: {signs_tamil}

Write EXACTLY ONE calm Tamil message explaining to the mother why the baby needs PHC immediately.
- Calm and clear — do not cause panic
- Explain what was found in simple Tamil
- Say baby needs care at PHC right now
- One action: go immediately, do not delay
- Maximum 3 sentences

Only Tamil. No English. No options."""

    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    return run_inference(messages, max_new_tokens=200)

# Test
print("TEST — Feeding + Warmth danger")
print(generate_referral_explanation(["feeding", "warmth"]))

In [ ]:
# Cell 9 — Feature A3: High Risk Baby Guidance
def generate_high_risk_guidance(baby_context):
    """
    Triggered when baby is flagged as high risk.
    Generates specific Tamil care instructions.
    """
    risk_reasons = []
    if baby_context.get('birth_weight', 3) < 2.0:
        risk_reasons.append(f"குழந்தை எடை {baby_context['birth_weight']} கிலோ மட்டுமே — மிகவும் குறைவு")
    if baby_context.get('is_premature'):
        risk_reasons.append("குழந்தை சீக்கிரமே பிறந்தது")
    if baby_context.get('home_delivery'):
        risk_reasons.append("வீட்டிலேயே பிறந்தது")
    if baby_context.get('birth_complications'):
        risk_reasons.append(f"பிறக்கும்போது சிக்கல்: {baby_context['birth_complications']}")
    
    risks_tamil = " | ".join(risk_reasons)
    
    prompt = f"""You are a health assistant for ASHA workers in Tamil Nadu.

This newborn is HIGH RISK: {risks_tamil}

Generate specific Tamil care instructions for the mother.
Focus on:
- Kangaroo Mother Care if low birth weight
- Extra warmth requirements
- More frequent feeding schedule  
- Specific danger signs for high risk babies
- How often ASHA will visit

Use simple informal Tamil. Maximum 6 sentences.
Be reassuring but clear about extra care needed.
Only Tamil. No English."""

    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    return run_inference(messages, max_new_tokens=350)

# Test
print("TEST — Premature + Low Birth Weight")
baby_ctx = {"birth_weight": 1.8, "is_premature": True, "home_delivery": True, "birth_complications": "born at 33 weeks"}
print(generate_high_risk_guidance(baby_ctx))

In [ ]:
# Cell 10 — Feature A4: Visit Summary for ANM Supervisor
def generate_visit_summary(visit_data):
    """
    Generates structured Tamil visit report for ANM supervisor.
    Replaces paper register entry.
    """
    danger_signs_map = {
        "feeding":   "பால் குடிக்கவில்லை",
        "activity":  "சுறுசுறுப்பு இல்லை",
        "warmth":    "உடல் குளிர்ச்சி",
        "breathing": "மூச்சு சரியில்லை"
    }
    
    if visit_data.get('danger_signs'):
        signs_text = ", ".join([danger_signs_map.get(s, s) for s in visit_data['danger_signs']])
        action = "PHC-க்கு அனுப்பப்பட்டது"
    else:
        signs_text = "எந்த அபாய அறிகுறியும் இல்லை"
        action = "வீட்டில் கவனிக்க ஆலோசிக்கப்பட்டது"
    
    prompt = f"""You are a health assistant for ASHA workers in Tamil Nadu.

Generate a formal Tamil visit summary report for an ANM supervisor.

Visit details:
- Baby: {visit_data['baby_name']} | Mother: {visit_data['mother_name']}
- Village: {visit_data['village']} | Date: {visit_data['date']}
- ASHA: {visit_data['asha_name']} | Visit Day: {visit_data['visit_day']}
- Birth weight: {visit_data['birth_weight']} kg
- Danger signs: {signs_text}
- Action: {action}
- Next visit: Day {visit_data['next_visit_day']}

Write a clean Tamil summary with these 4 sections:
1. குழந்தை விவரம் (Baby details)
2. வருகை மதிப்பீடு (Visit assessment)
3. எடுக்கப்பட்ட நடவடிக்கை (Action taken)
4. அடுத்த வருகை (Next visit)

Formal Tamil. Concise and professional."""

    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    return run_inference(messages, max_new_tokens=400)

# Test
print("TEST — Visit Summary with Referral")
vd = {
    "baby_name": "Baby Kavya", "mother_name": "Selvi",
    "village": "Pattabiram", "date": "26 April 2026",
    "asha_name": "Meena", "visit_day": 7,
    "birth_weight": 2.4, "danger_signs": ["feeding", "warmth"],
    "next_visit_day": 14
}
print(generate_visit_summary(vd))

In [ ]:
# Cell 11 — Feature B: Audio Processing (Mother's Concern Interpreter)
# Processes real Tamil audio from ASHA worker using Gemma 4 native audio understanding

import librosa
import subprocess
import json

def load_audio_file(file_path):
    """Convert any audio format to 16kHz mono wav and load as array"""
    wav_path = "/tmp/nalam_audio.wav"
    subprocess.run([
        'ffmpeg', '-i', file_path,
        '-ar', '16000', '-ac', '1',
        '-f', 'wav', wav_path,
        '-y', '-loglevel', 'quiet'
    ])
    audio_array, sr = librosa.load(wav_path, sr=16000)
    print(f"Audio loaded: {len(audio_array)/sr:.1f} seconds at {sr}Hz")
    return audio_array

def process_asha_audio(audio_array, baby_context):
    """
    Passes real Tamil audio directly into Gemma 4 E2B.
    Handles informal dialect, regional variation, code mixing.
    Returns structured clinical assessment as dict.
    """
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "audio",
                    "audio": audio_array,
                },
                {
                    "type": "text",
                    "text": f"""You are a clinical assistant for ASHA workers in rural Tamil Nadu.

The ASHA worker has spoken a concern about a newborn baby in Tamil.
She may use ANY dialect from Tamil Nadu — Chennai, Madurai, Coimbatore,
Tirunelveli, Salem, Dharmapuri, Vellore, Trichy, Thanjavur, or rural village dialect.

STEP 1: Transcribe exactly what she said.
STEP 2: Understand the MEANING, not just the words.

COLD / WARMTH expressions — ALL mean warmth concern:
- ஜில்லனு இருக்கு / ஜில்லுன்னு இருக்கு / பாடி ஜில்லனு இருக்கு
- body cold ஆ இருக்கு / குளுகுளுன்னு இருக்கு / குளுமையா இருக்கு
- சில்லுன்னு இருக்கு / ஒம்பு இல்லாம இருக்கு / உடம்பு தண்ணியா இருக்கு
- கை கால் குளிர்ச்சியா இருக்கு / heat இல்லை

NOT FEEDING expressions — ALL mean feeding concern:
- பால்யே குடிக்கலை / பாலே குடிக்கல / பால் குடிக்கவே மாட்டேங்குது
- வாயே திறக்கலை / feed பண்றதே இல்ல / குடிக்கவே மறுக்குது
- மார்பை பிடிக்கவே மாட்டா / suck பண்றதே இல்ல
- பசிக்கவே இல்லன்னு தெரியுது / தாய்ப்பால் வேணாம்னு திரும்புது

NOT ACTIVE expressions — ALL mean activity concern:
- சோம்பேறியா இருக்கு / சோம்பேறியா கிடக்கு / அசையவே மாட்டேங்குது
- தூக்கமா கிடக்கு / எழுப்பவே முடியல / ரெஸ்பான்ஸே இல்லை
- react பண்றதே இல்லை / dull ஆ இருக்கு / லேதர்ஜிக்கா இருக்கு
- கண்ணே தெரியல / உசிரே இல்லன்னு தெரியுது

BREATHING problems — ALL mean breathing concern:
- மூச்சு வேகமா விடுது / மூச்சு படபடக்குது / நெஞ்சு ஏறி இறங்குது
- chest ஏறி இறங்குது / மூச்சு முட்டுது / மூச்சு விட கஷ்டமா இருக்கு
- breathing fast ஆ இருக்கு / மூச்சு ஒழுங்கா இல்லை

Baby context:
- Visit day: {baby_context.get('visit_day')}
- Birth weight: {baby_context.get('birth_weight')} kg
- Premature: {baby_context.get('is_premature', False)}

Respond ONLY in this exact JSON:
{{
    "transcription": "exact words the ASHA said",
    "feeding_concern": true or false,
    "activity_concern": true or false,
    "warmth_concern": true or false,
    "breathing_concern": true or false,
    "urgency": "high" or "medium" or "low",
    "confidence": "high" or "medium" or "low",
    "reasoning": "one sentence in English explaining why you flagged these concerns",
    "tamil_followup_question": "clarifying question in Tamil if confidence is low, else empty string"
}}

Only the JSON. Nothing else."""
                }
            ]
        }
    ]
    
    # Run inference with explicit device placement
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True
    )
    
    # Move ALL tensors to cuda explicitly
    inputs = {
        k: v.to("cuda") if isinstance(v, torch.Tensor) else v
        for k, v in inputs.items()
    }
    
    input_len = inputs["input_ids"].shape[-1]
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=400,
            do_sample=False
        )
    
    response = processor.decode(
        outputs[0][input_len:],
        skip_special_tokens=True
    )
    
    # Parse JSON response
    clean_response = re.sub(r'```json|```', '', response).strip()
    json_match = re.search(r'\{.*?\}', clean_response, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group())
        except:
            return {"raw_response": response, "parse_error": True}
    return {"raw_response": response, "parse_error": True}

print("Audio processor ready")

In [ ]:
# Cell 12 — Test Audio Files
# Load and test q1, q2, q3 audio clips

baby_context = {
    "visit_day": 3,
    "birth_weight": 2.4,
    "is_premature": False
}

def print_audio_result(name, result):
    print(f"Transcription : {result.get('transcription')}")
    print(f"Feeding       : {result.get('feeding_concern')}")
    print(f"Activity      : {result.get('activity_concern')}")
    print(f"Warmth        : {result.get('warmth_concern')}")
    print(f"Breathing     : {result.get('breathing_concern')}")
    print(f"Urgency       : {result.get('urgency')}")
    print(f"Confidence    : {result.get('confidence')}")
    print(f"Reasoning     : {result.get('reasoning')}")
    if result.get('tamil_followup_question'):
        print(f"Followup      : {result.get('tamil_followup_question')}")

# Test Q1
print("=" * 50)
print("Q1")
print("=" * 50)
audio = load_audio_file("/kaggle/input/datasets/janetanand/nalama-audio/q1.mp4")
result = process_asha_audio(audio, baby_context)
print_audio_result("Q1", result)
print()

# Test Q2
print("=" * 50)
print("Q2")
print("=" * 50)
audio = load_audio_file("/kaggle/input/datasets/janetanand/nalama-audio/q2.mp4")
result = process_asha_audio(audio, baby_context)
print_audio_result("Q2", result)
print()

# Test Q3
print("=" * 50)
print("Q3")
print("=" * 50)
audio = load_audio_file("/kaggle/input/datasets/janetanand/nalama-audio/q3.mp4")
result = process_asha_audio(audio, baby_context)
print_audio_result("Q3", result)

In [ ]:
# Cell 13 — Complete Nalam Pipeline
# Combines rule-based decision with Gemma 4 communication

def nalam_assess(baby_name, visit_day, baby_weight_kg, answers, 
                 is_premature=False, home_delivery=False, audio_array=None):
    """
    Full Nalam assessment pipeline.
    
    answers: dict — feeding, activity, warmth, breathing
    True = normal, False = danger sign detected
    audio_array: optional — if ASHA recorded a voice concern
    """
    
    baby_context = {
        "visit_day": visit_day,
        "birth_weight": baby_weight_kg,
        "is_premature": is_premature
    }
    
    # Step 1 — Process voice concern if provided
    voice_flags = {}
    if audio_array is not None:
        print("Processing voice concern...")
        voice_flags = process_asha_audio(audio_array, baby_context)
        # Pre-flag any concerns detected from voice
        if voice_flags.get('feeding_concern') and answers.get('feeding', True):
            answers['feeding'] = False  # Override with voice finding
        if voice_flags.get('warmth_concern') and answers.get('warmth', True):
            answers['warmth'] = False
        if voice_flags.get('activity_concern') and answers.get('activity', True):
            answers['activity'] = False
        if voice_flags.get('breathing_concern') and answers.get('breathing', True):
            answers['breathing'] = False
    
    # Step 2 — Rule-based decision (WHO guidelines)
    # ANY one danger sign = refer immediately
    danger_signs_found = [k for k, v in answers.items() if v == False]
    is_safe = len(danger_signs_found) == 0
    
    # Step 3 — Gemma 4 generates Tamil communication
    if is_safe:
        tamil_message = generate_tamil_counselling(
            visit_day, baby_weight_kg, is_premature, home_delivery
        )
        decision = "MONITOR"
        decision_tamil = "வீட்டில் கவனிக்கவும்"
    else:
        tamil_message = generate_referral_explanation(danger_signs_found)
        decision = "REFER"
        decision_tamil = "உடனே PHC-க்கு அனுப்பவும்"
    
    # Step 4 — Generate visit summary
    import datetime
    visit_summary = generate_visit_summary({
        "baby_name": baby_name,
        "mother_name": "அம்மா",
        "village": "Village",
        "date": datetime.date.today().strftime("%d %B %Y"),
        "asha_name": "ASHA",
        "visit_day": visit_day,
        "birth_weight": baby_weight_kg,
        "danger_signs": danger_signs_found,
        "next_visit_day": {1:3, 3:7, 7:14, 14:21, 21:28, 28:42}.get(visit_day, 42)
    })
    
    return {
        "baby_name": baby_name,
        "visit_day": visit_day,
        "decision": decision,
        "decision_tamil": decision_tamil,
        "danger_signs": danger_signs_found,
        "tamil_message": tamil_message,
        "visit_summary": visit_summary,
        "is_safe": is_safe,
        "voice_transcription": voice_flags.get('transcription', None)
    }


# Full Pipeline Test 1 — Safe visit
print("FULL PIPELINE TEST 1 — Safe Visit")
print("=" * 50)
result1 = nalam_assess(
    baby_name="Baby Priya",
    visit_day=3,
    baby_weight_kg=2.8,
    answers={"feeding": True, "activity": True, "warmth": True, "breathing": True}
)
print(f"Decision: {result1['decision_tamil']}")
print(f"Message:\n{result1['tamil_message']}")
print()

# Full Pipeline Test 2 — Danger visit
print("FULL PIPELINE TEST 2 — Danger Visit")
print("=" * 50)
result2 = nalam_assess(
    baby_name="Baby Kavya",
    visit_day=7,
    baby_weight_kg=1.9,
    answers={"feeding": False, "activity": True, "warmth": False, "breathing": True},
    is_premature=True
)
print(f"Decision: {result2['decision_tamil']}")
print(f"Danger Signs: {result2['danger_signs']}")
print(f"Message:\n{result2['tamil_message']}")

## Hosting the Frontend\n1. Build your Vite React app using `npm run build` locally.\n2. Zip the `dist` folder created into `dist.zip`.\n3. Upload `dist.zip` to your Kaggle datasets and note the path.\n4. Unzip here using exactly the path where it was uploaded (e.g. `!unzip -o /kaggle/input/nalam-frontend/dist.zip -d /kaggle/working/dist`).\nOnce done, the Flask app below will serve it automatically!

In [ ]:
# Cell 14 — Flask API (contract-compliant)
# Endpoints, request bodies, and response shapes match API_CONTRACT.md.
# All decision-making is RULE-BASED; Gemma generates Tamil + English text only.

!pip install -q flask flask-cors pyngrok

import os
import base64
import json
from flask import Flask, request, jsonify, send_from_directory
from flask_cors import CORS

FRONTEND_DIST = '/kaggle/working/dist'   # optional: serve frontend from same origin
app = Flask(__name__, static_folder=FRONTEND_DIST if os.path.isdir(FRONTEND_DIST) else None,
            static_url_path='/')
CORS(app)


# ---- helpers ----------------------------------------------------------------

_DANGER_SIGN_TAMIL = {
    "feeding":   "குழந்தை சரியாக பால் குடிக்கவில்லை",
    "activity":  "குழந்தை சுறுசுறுப்பாக இல்லை",
    "warmth":    "குழந்தையின் உடல் குளிர்ச்சியாக உள்ளது",
    "breathing": "குழந்தை சரியாக மூச்சு விடவில்லை",
}
_DANGER_SIGN_ENGLISH = {
    "feeding":   "baby is not feeding properly",
    "activity":  "baby is not active",
    "warmth":    "baby's body is cold",
    "breathing": "baby is not breathing well",
}
_MOTHER_DANGER_TAMIL = {
    "bleeding":  "அதிக ரத்தப்போக்கு",
    "fever":     "காய்ச்சல்",
    "depressed": "தாயின் மன நிலை சரியில்லை",
}
_MOTHER_DANGER_ENGLISH = {
    "bleeding":  "heavy bleeding",
    "fever":     "fever",
    "depressed": "low mood / postpartum depression signs",
}


def _gemma_english_from_tamil(tamil_text):
    """Quick English rendering. Uses Gemma if available, falls back to a short label."""
    try:
        messages = [{"role": "user", "content": [{"type": "text",
            "text": f"Translate this Tamil message to short, plain English for an ASHA worker. "
                    f"Only the translation, no preamble.\n\n{tamil_text}"}]}]
        return run_inference(messages, max_new_tokens=200).strip()
    except Exception:
        return "See Tamil message for guidance."


# ---- /health ----------------------------------------------------------------

@app.route('/health', methods=['GET'])
def health():
    return jsonify({"status": "Nalam API running", "model": "Gemma 4 E2B"})


# ---- /assess ----------------------------------------------------------------

@app.route('/assess', methods=['POST'])
def assess():
    try:
        data = request.get_json(silent=True) or {}
        baby = data.get('baby') or {}
        answers = data.get('answers') or {}

        weight_kg = float(baby.get('weight') or 3.0)
        age_days = int(baby.get('age_days') or 3)
        premature = bool(baby.get('premature') or False)

        # Rule-based decision per contract (True = normal, False = danger)
        danger_signs_found = [k for k, v in answers.items() if v is False]
        is_safe = len(danger_signs_found) == 0

        if is_safe:
            tamil_message = generate_tamil_counselling(
                visit_day=age_days, baby_weight_kg=weight_kg,
                is_premature=premature, home_delivery=False
            )
        else:
            tamil_message = generate_referral_explanation(danger_signs_found)

        english_message = _gemma_english_from_tamil(tamil_message)

        return jsonify({
            "is_safe": is_safe,
            "tamil_message": tamil_message,
            "english_message": english_message,
            "danger_signs": danger_signs_found,
        })
    except Exception as e:
        signs_t = " மற்றும் ".join(_DANGER_SIGN_TAMIL.values())
        return jsonify({
            "is_safe": False,
            "tamil_message": "மதிப்பீடு தோல்வியடைந்தது. பாதுகாப்புக்காக குழந்தையை உடனே PHC-க்கு அனுப்பவும்.",
            "english_message": "Assessment failed. For safety, refer the baby to PHC immediately.",
            "danger_signs": ["unknown_risk_fallback"],
        })


# ---- /mother-health ---------------------------------------------------------

@app.route('/mother-health', methods=['POST'])
def mother_health():
    try:
        data = request.get_json(silent=True) or {}
        answers = data.get('answers') or {}
        danger_signs_found = [k for k, v in answers.items() if v is True]
        is_safe = len(danger_signs_found) == 0

        if is_safe:
            tamil_message = (
                "தாய்க்கு இப்போது எந்த அபாய அறிகுறியும் இல்லை. ஓய்வு எடுக்கவும், "
                "சத்தான உணவு உண்ணவும், அதிக நீர் குடிக்கவும்."
            )
        else:
            signs_t = " மற்றும் ".join(
                [_MOTHER_DANGER_TAMIL.get(s, s) for s in danger_signs_found]
            )
            tamil_message = (
                f"தாய்க்கு {signs_t} உள்ளது. இது அவசர நிலை. உடனே PHC-க்கு "
                "அழைத்து செல்லவும். தேவைப்பட்டால் 108 அழைக்கவும்."
            )

        english_message = _gemma_english_from_tamil(tamil_message)

        return jsonify({
            "is_safe": is_safe,
            "tamil_message": tamil_message,
            "english_message": english_message,
            "danger_signs": danger_signs_found,
        })
    except Exception:
        return jsonify({
            "is_safe": False,
            "tamil_message": "தாய் மதிப்பீடு தோல்வியடைந்தது. பாதுகாப்புக்காக PHC-க்கு அனுப்பவும்.",
            "english_message": "Mother-health assessment failed. For safety, refer to PHC.",
            "danger_signs": ["unknown_maternal_risk_fallback"],
        })


# ---- /audio-text ------------------------------------------------------------

@app.route('/audio-text', methods=['POST'])
def audio_text():
    """
    Browser already did Tamil STT; backend extracts concern flags from the text.
    Uses Gemma to read the transcript and decide which checklist items to flag.
    """
    try:
        data = request.get_json(silent=True) or {}
        text = (data.get('text') or '').strip()
        baby_context = data.get('baby_context') or {}
        visit_day = baby_context.get('visit_day', 3)
        birth_weight = baby_context.get('birth_weight', 'unknown')

        prompt = (
            f"You are a Tamil HBNC assistant. The ASHA worker reports the mother's concern about a "
            f"day-{visit_day}, {birth_weight} kg newborn:\n\n"
            f"\"{text}\"\n\n"
            "Return a strict JSON object (no prose) with keys: "
            "feeding_concern, activity_concern, warmth_concern, breathing_concern (booleans), "
            "urgency ('low' or 'high'), confidence ('low'|'medium'|'high'), reasoning (one sentence). "
            "Flag a concern only if the text clearly implies it."
        )
        messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
        raw = run_inference(messages, max_new_tokens=250).strip()

        # Strip code fences if Gemma added them
        if raw.startswith("```"):
            raw = raw.strip("`")
            if raw.lower().startswith("json"):
                raw = raw[4:].strip()
        parsed = json.loads(raw)

        return jsonify({
            "transcription": text,
            "feeding_concern":   bool(parsed.get("feeding_concern", False)),
            "activity_concern":  bool(parsed.get("activity_concern", False)),
            "warmth_concern":    bool(parsed.get("warmth_concern", False)),
            "breathing_concern": bool(parsed.get("breathing_concern", False)),
            "urgency":    parsed.get("urgency", "low"),
            "confidence": parsed.get("confidence", "medium"),
            "reasoning":  parsed.get("reasoning", ""),
        })
    except Exception as e:
        return jsonify({
            "transcription": (data.get('text') if isinstance(data, dict) else '') or "",
            "feeding_concern": True, "activity_concern": True,
            "warmth_concern": False, "breathing_concern": False,
            "urgency": "high", "confidence": "low",
            "reasoning": "Audio-text processing failed; conservative concern fallback applied."
        })


# ---- /counselling -----------------------------------------------------------

@app.route('/counselling', methods=['POST'])
def counselling():
    try:
        data = request.get_json(silent=True) or {}
        baby = data.get('baby')
        danger_signs = data.get('danger_signs')
        is_safe = data.get('is_safe')
        mother_result = data.get('mother_result')

        if not isinstance(baby, dict) or not isinstance(danger_signs, list) \
                or not isinstance(is_safe, bool) or not isinstance(mother_result, dict):
            raise ValueError("invalid body")

        weight_kg = float(baby.get('weight') or 3.0)
        age_days = baby.get('age_days') if isinstance(baby.get('age_days'), int) else 3
        premature = bool(baby.get('premature') or False)
        mother_is_safe = bool(mother_result.get('is_safe'))

        if not is_safe and danger_signs:
            action = "REFER_PHC" if mother_is_safe else "REFER_108"
            tamil = generate_referral_explanation(danger_signs)
        elif not mother_is_safe:
            action = "REFER_PHC"
            tamil = (
                "தாய்க்கு அபாய அறிகுறிகள் உள்ளன. தாயை உடனே PHC-க்கு கொண்டு செல்லவும். "
                "குழந்தைக்கு தாய்ப்பால் வழியில் தொடரவும்."
            )
        else:
            action = "HOME_CARE"
            tamil = generate_tamil_counselling(
                visit_day=age_days, baby_weight_kg=weight_kg,
                is_premature=premature, home_delivery=False
            )

        english = _gemma_english_from_tamil(tamil)

        return jsonify({
            "tamil_counselling_text": tamil,
            "english_counselling_text": english,
            "action": action,
        })
    except Exception:
        return jsonify({
            "tamil_counselling_text": "ஆலோசனை உருவாக்க முடியவில்லை. பாதுகாப்புக்காக உடனே 108 அழைக்கவும்.",
            "english_counselling_text": "Counselling generation failed. For safety, call 108 now.",
            "action": "REFER_108",
        })


# ---- frontend passthrough (optional) ----------------------------------------

@app.route('/', defaults={'path': ''})
@app.route('/<path:path>')
def serve(path):
    if not app.static_folder:
        return jsonify({"status": "Nalam API running", "model": "Gemma 4 E2B"})
    if path and os.path.exists(os.path.join(app.static_folder, path)):
        return send_from_directory(app.static_folder, path)
    return send_from_directory(app.static_folder, 'index.html')


# ---- start ngrok + Flask ----------------------------------------------------

print("Flask app defined. Starting ngrok tunnel...")
from pyngrok import ngrok
ngrok.kill()
http_tunnel = ngrok.connect(5000)
print(f"\nPublic URL: {http_tunnel.public_url}")
print("Set VITE_API_URL=<that URL> in Vercel, then redeploy the frontend.")
print("\nStarting Flask server (this blocks — keep the notebook running)...")
app.run(port=5000, debug=False, use_reloader=False)


## API Endpoints (matches API_CONTRACT.md)

| Endpoint | Method | Purpose |
|---|---|---|
| `/health` | GET | Service health |
| `/assess` | POST | Newborn checklist assessment |
| `/mother-health` | POST | Maternal danger-sign assessment |
| `/audio-text` | POST | Extract concerns from Tamil transcript |
| `/counselling` | POST | Generate Tamil + English counselling script |

All endpoints follow the canonical request/response shapes in `API_CONTRACT.md`.
Rule-based decision-making lives in the Flask layer; Gemma is used **only** for
generating warm Tamil text (and the English mirror via translation).

### Quick test
```bash
curl -X POST $PUBLIC_URL/assess -H 'Content-Type: application/json' -d '{
  "baby": {"name": "Baby Priya", "weight": "2.8", "age_days": 3, "premature": false},
  "answers": {"feeding": true, "activity": true, "warmth": true, "breathing": true}
}'
```
